In [ ]:
import numpy as np
import os
import pywt
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert, welch
import pandas as pd
from multiprocessing import Pool, cpu_count

# Define the directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\term-preterm-ehg-database-1.0.1\preprocessed_ehg"

# Function to calculate time-domain features
def time_domain_features(data):
    mean = np.mean(data)
    variance = np.var(data)
    skewness = skew(data)
    kurt = kurtosis(data)
    rms = np.sqrt(np.mean(data ** 2))
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    zero_crossings = np.sum(np.diff(np.sign(data)) != 0)
    envelope = np.max(data) - np.min(data)
    return [mean, variance, skewness, kurt, rms, iqr, zero_crossings, envelope]

# Function to calculate sample and permutation entropy
def entropy_features(data):
    def sample_entropy(data, m=2, r=0.2):
        n = len(data)
        if n <= m:
            return np.nan
        def _phi(m):
            x = np.array([data[i:i + m] for i in range(n - m + 1)])
            C = np.array([np.sum(np.max(np.abs(x - xi), axis=1) < r) for xi in x])
            return np.sum(C) / (n - m + 1) if (n - m + 1) > 0 else 0
        phi_m = _phi(m)
        phi_m_plus_1 = _phi(m + 1)
        if phi_m > 0 and phi_m_plus_1 > 0:
            return -np.log(phi_m_plus_1 / phi_m)
        else:
            return np.nan
    def permutation_entropy(data, m=3):
        from itertools import permutations
        n = len(data)
        permutations_dict = {p: i for i, p in enumerate(permutations(range(m)))}
        patterns = [tuple(np.argsort(data[i:i + m])) for i in range(n - m + 1)]
        return -np.log(np.bincount([permutations_dict[pattern] for pattern in patterns]).max() / (n - m + 1))
    sample_ent = sample_entropy(data)
    perm_ent = permutation_entropy(data)
    return [sample_ent, perm_ent]

# Function to calculate frequency-domain features
def frequency_domain_features(data, fs=20):
    f, psd = welch(data, fs=fs, nperseg=256)
    psd_band_0_3_0_6 = np.trapz(psd[(f >= 0.3) & (f < 0.6)])
    psd_band_0_6_1_0 = np.trapz(psd[(f >= 0.6) & (f < 1.0)])
    dominant_frequency = f[np.argmax(psd)]
    dominant_frequency_ratio = np.max(psd) / np.sum(psd)
    psd_normalized = psd / np.sum(psd)
    spectral_entropy = -np.sum(psd_normalized * np.log(psd_normalized + np.finfo(float).eps))
    return [psd_band_0_3_0_6, psd_band_0_6_1_0, dominant_frequency, dominant_frequency_ratio, spectral_entropy]

# Function to calculate wavelet transform features
def wavelet_transform_features(data, wavelet='db4', levels=5):
    coeffs = pywt.wavedec(data, wavelet, level=levels)
    wavelet_energy = [np.sum(c ** 2) for c in coeffs]
    return wavelet_energy

# Function to calculate Hilbert transform features
def hilbert_transform_features(data):
    analytic_signal = hilbert(data)
    hilbert_amplitude = np.mean(np.abs(analytic_signal))
    hilbert_frequency = np.mean(np.diff(np.unwrap(np.angle(analytic_signal)))) / (2.0 * np.pi)
    hilbert_frequency_std = np.std(np.diff(np.unwrap(np.angle(analytic_signal)))) / (2.0 * np.pi)
    return [hilbert_amplitude, hilbert_frequency, hilbert_frequency_std]

# Main feature extraction function for a single file
def extract_features_for_parallel(file_path):
    data = np.fromfile(file_path, dtype=np.float32)
    features = (
        time_domain_features(data)
        + entropy_features(data)
        + frequency_domain_features(data)
        + wavelet_transform_features(data)
        + hilbert_transform_features(data)
    )
    return os.path.basename(file_path), features

# Parallel processing function
def process_files_in_parallel(files):
    file_paths = [os.path.join(preprocessed_dir, f) for f in files]
    with Pool(cpu_count()) as pool:
        results = pool.map(extract_features_for_parallel, file_paths)
    return results

# Collect all preprocessed files
files = [f for f in os.listdir(preprocessed_dir) if f.startswith('tpehg')]

# Process files in parallel
results = process_files_in_parallel(files)

# Create a DataFrame to hold the features
columns = [
    'File Name',
    'Mean', 'Variance', 'Skewness', 'Kurtosis', 'RMS', 'IQR', 'Zero Crossings', 'Envelope',
    'Sample Entropy', 'Permutation Entropy',
    'PSD Band 0.3–0.6 Hz', 'PSD Band 0.6–1.0 Hz', 'Dominant Frequency', 'Dominant Frequency Ratio', 'Spectral Entropy',
    'Wavelet Energy Level 0', 'Wavelet Energy Level 1', 'Wavelet Energy Level 2', 'Wavelet Energy Level 3',
    'Wavelet Energy Level 4', 'Wavelet Energy Level 5',
    'Hilbert Amplitude Mean', 'Hilbert Frequency Mean', 'Hilbert Frequency Std'
]

# Convert results to DataFrame
features_df = pd.DataFrame(
    [([file_name] + features) for file_name, features in results], 
    columns=columns
)

# Save features to a CSV file
output_csv_path = os.path.join(preprocessed_dir, 'extracted_features.csv')
features_df.to_csv(output_csv_path, index=False)

print(f"Extracted features saved to {output_csv_path}")


# Time domain features

In [1]:
import numpy as np
import os
import pandas as pd
from scipy.stats import skew, kurtosis

# Define the directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco"

# Function to calculate time-domain features
def time_domain_features(data):
    mean = np.mean(data)
    variance = np.var(data)
    skewness = skew(data)
    kurt = kurtosis(data)
    rms = np.sqrt(np.mean(data ** 2))
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    zero_crossings = np.sum(np.diff(np.sign(data)) != 0)
    envelope = np.max(data) - np.min(data)
    
    return [mean, variance, skewness, kurt, rms, iqr, zero_crossings, envelope]

# Function to extract time-domain features from a file
def extract_time_domain_features_from_file(file_path):
    # Load the preprocessed data
    data = np.fromfile(file_path, dtype=np.float32)

    # Assuming the data is structured channel-wise
    num_channels = 3  # Correct number of channels
    channel_length = len(data) // num_channels
    features = []

    for channel in range(num_channels):
        channel_data = data[channel * channel_length : (channel + 1) * channel_length]
        time_features = time_domain_features(channel_data)
        features.append(time_features)

    return features

# Process all preprocessed files in the directory and save time-domain features to a DataFrame
time_feature_list = []
files = [f for f in os.listdir(preprocessed_dir) if f.startswith('tpehg') and f.endswith('.dat')]
for file_name in files:
    file_path = os.path.join(preprocessed_dir, file_name)
    channel_features = extract_time_domain_features_from_file(file_path)

    # Remove .dat extension for the Record column
    record_name = file_name.replace('.dat', '')
    
    # Append features channel-wise
    for channel_index, features in enumerate(channel_features, start=1):
        time_feature_list.append([record_name, channel_index] + features)

# Create a DataFrame to hold the time-domain features
time_columns = [
    'Record', 'Channel',
    'Mean', 'Variance', 'Skewness', 'Kurtosis', 'RMS', 'IQR', 'Zero Crossings', 'Envelope'
]

time_features_df = pd.DataFrame(time_feature_list, columns=time_columns)

# Save time-domain features to a CSV file
output_csv_path = 'time_domain_features.csv'
time_features_df.to_csv(output_csv_path, index=False)

print(f"Extracted time-domain features saved to {output_csv_path}")


Extracted time-domain features saved to time_domain_features.csv


# frequency domain features

In [2]:
import numpy as np
import os
import pandas as pd
from scipy.signal import welch

# Define the directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco"

# Function to calculate spectral entropy
def spectral_entropy(psd, psd_sum):
    psd_norm = psd / psd_sum  # Normalize the PSD
    return -np.sum(psd_norm * np.log2(psd_norm + 1e-10))  # Add small value to avoid log(0)

# Function to calculate frequency-domain features
def frequency_domain_features(data, fs=20):
    # Calculate the Power Spectral Density (PSD)
    f, psd = welch(data, fs=fs, nperseg=256)
    psd_sum = np.sum(psd)  # Total power for normalization

    # Calculate features from PSD
    psd_band_0_3 = np.trapz(psd[(f >= 0.3) & (f < 0.6)])  # Power in 0.3-0.6 Hz band
    psd_band_0_6 = np.trapz(psd[(f >= 0.6) & (f < 1.0)])  # Power in 0.6-1.0 Hz band
    dominant_frequency = f[np.argmax(psd)]                 # Frequency with max power
    mean_frequency = np.sum(f * psd) / psd_sum             # Mean frequency
    total_power = psd_sum                                  # Total power of the signal
    spec_entropy = spectral_entropy(psd, psd_sum)          # Spectral entropy

    return [psd_band_0_3, psd_band_0_6, dominant_frequency, mean_frequency, total_power, spec_entropy]

# Function to extract frequency-domain features from a file
def extract_frequency_domain_features_from_file(file_path):
    # Load the preprocessed data
    data = np.fromfile(file_path, dtype=np.float32)

    # Return the loaded data
    return data

# Process all preprocessed files in the directory and save frequency-domain features to a DataFrame
freq_feature_list = []
files = [f for f in os.listdir(preprocessed_dir) if f.startswith('tpehg')]
for file_name in files:
    file_path = os.path.join(preprocessed_dir, file_name)
    # Extract data from file
    data = extract_frequency_domain_features_from_file(file_path)
    
    # Extract frequency-domain features for each channel
    num_channels = 3  # Assuming 3 channels
    samples_per_channel = len(data) // num_channels
    for channel in range(num_channels):
        channel_data = data[channel * samples_per_channel:(channel + 1) * samples_per_channel]  # Adjust according to your data structure
        freq_features = frequency_domain_features(channel_data)  # Directly calculate features from channel data
        freq_feature_list.append([file_name[:-4], channel + 1] + freq_features)  # Remove .dat extension

# Create a DataFrame to hold the frequency-domain features
freq_columns = [
    'Record',
    'Channel',
    'PSD Band 0.3–0.6 Hz', 'PSD Band 0.6–1.0 Hz', 'Dominant Frequency', 
    'Mean Frequency', 'Total Power', 'Spectral Entropy'
]

frequency_features_df = pd.DataFrame(freq_feature_list, columns=freq_columns)

# Save frequency-domain features to a CSV file
output_freq_csv_path = 'frequency_domain_features.csv'
frequency_features_df.to_csv(output_freq_csv_path, index=False)

print(f"Extracted frequency-domain features (including spectral entropy) saved to {output_freq_csv_path}")


Extracted frequency-domain features (including spectral entropy) saved to frequency_domain_features.csv


# Non Linear features

In [ ]:
import os
import numpy as np
import pandas as pd
import nolds
import time
from concurrent.futures import ProcessPoolExecutor
import warnings

# Suppress runtime warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco"

# Helper Functions
def normalize_data(data):
    return (data - np.mean(data)) / np.std(data)

def calculate_sample_entropy(data):
    try:
        normalized_data = normalize_data(data)
        return nolds.sampen(normalized_data)
    except Exception:
        return np.nan

def calculate_approximate_entropy(data):
    try:
        normalized_data = normalize_data(data)
        return nolds.apen(normalized_data, emb_dim=2)
    except Exception:
        return np.nan

# Function to process a single file
def process_file(file_path):
    try:
        data = np.fromfile(file_path, dtype=np.float32)
        num_channels = 3
        samples_per_channel = len(data) // num_channels
        file_name = os.path.basename(file_path)

        features = []
        for channel in range(num_channels):
            channel_data = data[channel * samples_per_channel:(channel + 1) * samples_per_channel]

            features.append({
                "Record": file_name[:-4],
                "Channel": channel + 1,
                "Sample Entropy": calculate_sample_entropy(channel_data),
                "Approximate Entropy": calculate_approximate_entropy(channel_data),
            })

        return features
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return []

# Main function with parallel processing
def main():
    start_time = time.time()

    # Collect all file paths
    files = [os.path.join(preprocessed_dir, f) for f in os.listdir(preprocessed_dir) if f.startswith('tpehg')]
    total_files = len(files)

    print(f"Starting feature extraction for {total_files} files...")
    all_features = []

    # Process files in parallel
    with ProcessPoolExecutor() as executor:
        for idx, file_features in enumerate(executor.map(process_file, files), start=1):
            all_features.extend(file_features)

            # Progress logging
            elapsed_time = time.time() - start_time
            estimated_time_remaining = (elapsed_time / idx) * (total_files - idx)
            print(f"Processed {idx}/{total_files} files. "
                  f"Estimated time remaining: {estimated_time_remaining:.2f} seconds.")

    # Save all features to a CSV
    output_csv_path = 'optimized_nonlinear_features.csv'
    features_df = pd.DataFrame(all_features)
    features_df.to_csv(output_csv_path, index=False)

    print(f"Feature extraction completed. Results saved to {output_csv_path}.")
    print(f"Total time taken: {time.time() - start_time:.2f} seconds.")

if __name__ == "__main__":
    main()


In [3]:
import os
import numpy as np
import pandas as pd
import logging
import warnings
import time

# Suppress runtime warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Logging configuration
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger()

# Directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco"
MAX_FILE_SIZE_MB = 100  # Skip files larger than 100 MB

# Helper Functions
def normalize_data(data):
    return (data - np.mean(data)) / np.std(data)

def calculate_rqa_features(data):
    try:
        # Placeholder for RQA calculation
        rr = np.random.random()  # Recurrence Rate (placeholder)
        det = np.random.random()  # Determinism (placeholder)
        lam = np.random.random()  # Laminarity (placeholder)
        lmax = np.random.random()  # Lmax (placeholder)
        return {
            "Recurrence Rate": rr,
            "Determinism": det,
            "Laminarity": lam,
            "L_max": lmax,
        }
    except Exception as e:
        logger.error(f"Error in RQA calculation: {e}")
        return {
            "Recurrence Rate": np.nan,
            "Determinism": np.nan,
            "Laminarity": np.nan,
            "L_max": np.nan,
        }

def process_file(file_path):
    """Processes a single file and extracts RQA features for each channel."""
    try:
        if os.path.getsize(file_path) > MAX_FILE_SIZE_MB * 1024 * 1024:
            logger.warning(f"Skipping large file: {file_path}")
            return []
        
        logger.info(f"Processing file: {file_path}")
        data = np.fromfile(file_path, dtype=np.float32)

        if len(data) == 0:
            raise ValueError(f"File {file_path} is empty.")

        num_channels = 3
        samples_per_channel = len(data) // num_channels
        file_name = os.path.basename(file_path)

        features = []
        for channel in range(num_channels):
            channel_data = data[channel * samples_per_channel:(channel + 1) * samples_per_channel]
            rqa_features = calculate_rqa_features(channel_data)
            rqa_features.update({
                "Record": file_name[:-4],
                "Channel": channel + 1,
            })
            features.append(rqa_features)

        return features
    except Exception as e:
        logger.error(f"Error processing file {file_path}: {e}")
        return []

def main():
    start_time = time.time()

    # Collect all file paths
    files = [os.path.join(preprocessed_dir, f) for f in os.listdir(preprocessed_dir) if f.startswith('tpehg')]
    total_files = len(files)
    logger.info(f"Found {total_files} files for processing.")

    all_features = []
    for idx, file_path in enumerate(files, start=1):
        file_features = process_file(file_path)
        all_features.extend(file_features)

        # Progress Logging
        elapsed_time = time.time() - start_time
        estimated_time_remaining = (elapsed_time / idx) * (total_files - idx)
        logger.info(f"Processed {idx}/{total_files} files. "
                    f"Estimated time remaining: {estimated_time_remaining:.2f} seconds.")

    # Save features to CSV
    output_csv_path = 'rqa_features_sequential.csv'
    features_df = pd.DataFrame(all_features)
    features_df.to_csv(output_csv_path, index=False)
    logger.info(f"Feature extraction completed. Results saved to {output_csv_path}.")
    logger.info(f"Total time taken: {time.time() - start_time:.2f} seconds.")

if __name__ == "__main__":
    main()


2025-02-23 10:19:51,068 - Found 31 files for processing.
2025-02-23 10:19:51,070 - Processing file: C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco\tpehgt_n001.dat
2025-02-23 10:19:51,072 - Processed 1/31 files. Estimated time remaining: 0.12 seconds.
2025-02-23 10:19:51,074 - Processing file: C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco\tpehgt_n002.dat
2025-02-23 10:19:51,076 - Processed 2/31 files. Estimated time remaining: 0.12 seconds.
2025-02-23 10:19:51,077 - Processing file: C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco\tpehgt_n003.dat
2025-02-23 10:19:51,079 - Processed 3/31 files. Estimated time remaining: 0.11 seconds.
2025-02-23 10:19:51,080 - Processing file: C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco\tpehgt_n004.dat
2025-02-23 10:19:51,083 - Processed 4/31 files. Estimated time remaining: 0.10 seconds.
2025-02-23 10:19:51,084

In [1]:
pip install pyrqa

Note: you may need to restart the kernel to use updated packages.


# Time-Frequency Features

In [4]:
import numpy as np
import os
import pywt
from scipy.signal import hilbert
from scipy.stats import entropy
import pandas as pd

# Define the directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco"

# Function to calculate wavelet energy and entropy
def wavelet_features(data, wavelet='db4', levels=5):
    coeffs = pywt.wavedec(data, wavelet, level=levels)
    energies = [np.sum(np.square(c)) for c in coeffs]  # Wavelet energy
    total_energy = sum(energies)
    normalized_energies = [e / total_energy for e in energies]
    wavelet_entropy = entropy(normalized_energies)  # Wavelet entropy

    features = {
        
        **{f'Wavelet Energy Level {i+1}': energies[i] for i in range(len(energies))},
        'Wavelet Entropy': wavelet_entropy
    }
    return features

# Function to calculate instantaneous amplitude and frequency
def hilbert_features(data):
    analytic_signal = hilbert(data)
    instantaneous_amplitude = np.abs(analytic_signal)
    instantaneous_phase = np.unwrap(np.angle(analytic_signal))
    instantaneous_frequency = np.diff(instantaneous_phase) / (2.0 * np.pi)

    features = {
        'Instantaneous Amplitude Mean': np.mean(instantaneous_amplitude),
        'Instantaneous Amplitude Std': np.std(instantaneous_amplitude),
        'Instantaneous Frequency Mean': np.mean(instantaneous_frequency),
        'Instantaneous Frequency Std': np.std(instantaneous_frequency)
    }
    return features

# Extract time-frequency features from a file for all channels
def extract_time_frequency_features_from_file(file_path, num_channels=3):
    # Load the preprocessed data
    data = np.fromfile(file_path, dtype=np.float32)
    
    # Ensure data is divisible by the number of channels
    samples_per_channel = len(data) // num_channels
    data = data[:samples_per_channel * num_channels]  # Trim excess samples if any

    # Extract features for each channel
    all_features = []
    for channel in range(num_channels):
        channel_data = data[channel * samples_per_channel:(channel + 1) * samples_per_channel]
        
        # Extract features
        wavelet_feats = wavelet_features(channel_data)
        hilbert_feats = hilbert_features(channel_data)

        # Combine features and add channel info
        features = {**wavelet_feats, **hilbert_feats}
        features['Channel'] = channel + 1
        all_features.append(features)

    return all_features

# Process all preprocessed files and save features
time_freq_features_list = []
files = [f for f in os.listdir(preprocessed_dir) if f.startswith('tpehg')]
for file_name in files:
    file_path = os.path.join(preprocessed_dir, file_name)
    
    # Extract features for all channels
    features_per_channel = extract_time_frequency_features_from_file(file_path)
    for channel_features in features_per_channel:
        channel_features['Record'] = file_name[:-4]  # Remove file extension
        time_freq_features_list.append(channel_features)

# Create a DataFrame to hold the time-frequency features
time_freq_features_df = pd.DataFrame(time_freq_features_list)

# Define the desired column order
desired_column_order = [
    'Record', 'Channel',
    'Wavelet Energy Level 1', 'Wavelet Energy Level 2', 'Wavelet Energy Level 3',
    'Wavelet Energy Level 4', 'Wavelet Energy Level 5', 'Wavelet Entropy',
    'Instantaneous Amplitude Mean', 'Instantaneous Amplitude Std',
    'Instantaneous Frequency Mean', 'Instantaneous Frequency Std'
]

# Reorder the DataFrame columns
time_freq_features_df = time_freq_features_df[desired_column_order]

# Save features to a CSV file
output_csv_path = 'time_frequency_features.csv'
time_freq_features_df.to_csv(output_csv_path, index=False)

print(f"Extracted time-frequency features saved to {output_csv_path}")


Extracted time-frequency features saved to time_frequency_features.csv


In [7]:
import pandas as pd

# Load the extracted feature files
time_df = pd.read_csv('time_domain_features.csv')
freq_df = pd.read_csv('frequency_domain_features.csv')
rqa_df = pd.read_csv('rqa_features.csv')
wavelet_df = pd.read_csv('time_frequency_features.csv')  # Replace with the correct file name

# Merge all feature datasets on Record and Channel
merged_df = time_df.merge(freq_df, on=['Record', 'Channel'], how='inner')
merged_df = merged_df.merge(rqa_df, on=['Record', 'Channel'], how='inner')
merged_df = merged_df.merge(wavelet_df, on=['Record', 'Channel'], how='inner')

# Select only the required columns in the given order
final_columns = [
    'Record', 'Channel', 'Mean', 'Variance', 'Skewness', 'Kurtosis', 'RMS', 'IQR',
    'Zero Crossings', 'Envelope', 'PSD Band 0.3–0.6 Hz', 'PSD Band 0.6–1.0 Hz', 'Mean Frequency', 
    'Spectral Entropy', 'Wavelet Energy Level 1', 'Wavelet Energy Level 3', 'Wavelet Entropy',
    'Instantaneous Amplitude Std', 'Instantaneous Frequency Mean', 'Instantaneous Frequency Std', 
    'Recurrence Rate', 'Determinism', 'Laminarity', 'L_max'
]

# Ensure only selected columns are included
final_df = merged_df[final_columns]

# Save to an Excel file
final_df.to_csv('Final_Features.csv', index=False)

print("Final feature set saved as 'Final_Features.csv'")


Final feature set saved as 'Final_Features.csv'


In [8]:
import os
import numpy as np
import pandas as pd
import nolds
import time
import warnings

# Suppress runtime warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Directory containing the preprocessed files
preprocessed_dir = r"C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram\preprocessed_ehg_toco"

# Helper Function: Normalize Data
def normalize_data(data):
    return (data - np.mean(data)) / np.std(data)

# Function to calculate Sample Entropy
def calculate_sample_entropy(data):
    try:
        normalized_data = normalize_data(data)
        return nolds.sampen(normalized_data)
    except Exception:
        return np.nan

# Function to process a single file
def process_file(file_path):
    try:
        data = np.fromfile(file_path, dtype=np.float32)
        num_channels = 3
        samples_per_channel = len(data) // num_channels
        file_name = os.path.basename(file_path)

        features = []
        for channel in range(num_channels):
            channel_data = data[channel * samples_per_channel:(channel + 1) * samples_per_channel]

            features.append({
                "Record": file_name[:-4],
                "Channel": channel + 1,
                "Sample Entropy": calculate_sample_entropy(channel_data),
            })

        return features
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return []

# Main function (Sequential Execution)
def main():
    start_time = time.time()

    # Collect all file paths
    files = [os.path.join(preprocessed_dir, f) for f in os.listdir(preprocessed_dir) if f.startswith('tpehg')]
    total_files = len(files)

    print(f"Starting Sample Entropy extraction for {total_files} files...")
    all_features = []

    # Process files one by one
    for idx, file_path in enumerate(files, start=1):
        file_features = process_file(file_path)
        all_features.extend(file_features)

        # Progress Logging
        elapsed_time = time.time() - start_time
        estimated_time_remaining = (elapsed_time / idx) * (total_files - idx)
        print(f"Processed {idx}/{total_files} files. "
              f"Estimated time remaining: {estimated_time_remaining:.2f} seconds.")

    # Save results to CSV
    output_csv_path = 'sample_entropy_features.csv'
    features_df = pd.DataFrame(all_features)
    features_df.to_csv(output_csv_path, index=False)

    print(f"Feature extraction completed. Results saved to {output_csv_path}.")
    print(f"Total time taken: {time.time() - start_time:.2f} seconds.")

if __name__ == "__main__":
    main()


Starting Sample Entropy extraction for 31 files...
Processed 1/31 files. Estimated time remaining: 26601.42 seconds.
Processed 2/31 files. Estimated time remaining: 24751.04 seconds.
Processed 3/31 files. Estimated time remaining: 23457.12 seconds.
Processed 4/31 files. Estimated time remaining: 22483.61 seconds.
Processed 5/31 files. Estimated time remaining: 21527.47 seconds.
Processed 6/31 files. Estimated time remaining: 20710.83 seconds.
Processed 7/31 files. Estimated time remaining: 19845.41 seconds.
Processed 8/31 files. Estimated time remaining: 19044.15 seconds.
Processed 9/31 files. Estimated time remaining: 18252.71 seconds.
Processed 10/31 files. Estimated time remaining: 17406.91 seconds.
Processed 11/31 files. Estimated time remaining: 16606.85 seconds.
Processed 12/31 files. Estimated time remaining: 16168.30 seconds.
Processed 13/31 files. Estimated time remaining: 15647.23 seconds.
Processed 14/31 files. Estimated time remaining: 15062.67 seconds.
Processed 15/31 file